In [1]:
"""
================================================================================
COMPARATIVE ANALYSIS: CRYPTOGUARD-MLP vs LINEAR REGRESSION vs LOGISTIC REGRESSION vs SVM vs XGBoost vs LightGBM vs CatBoost
5-Fold Stratified CV with proper train/val/test split (14.4k/1.6k/4k per fold)
Early Stopping on Validation Data for All Models
Combined Curves: GREEN=MLP, RED=Linear Regression, ORANGE=Logistic Regression (μ ± σ bands, min_epochs)
Tree Models: Early Stopping Enabled (XGBoost, LightGBM, CatBoost with 500 iterations)
================================================================================
"""

# ============ SYSTEM & UTILITIES ============
import os
import sys
import gc
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ============ DEEP LEARNING (MLP) ============
import tensorflow as tf
from tensorflow import keras
from keras import Sequential, regularizers
from keras.layers import Dense, Dropout, Input, BatchNormalization
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ============ SKLEARN - DATA & METRICS ============
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (matthews_corrcoef, accuracy_score, precision_score, 
                             recall_score, f1_score, confusion_matrix)

# ============ VISUALIZATION ============
import matplotlib.pyplot as plt

# ============ RANDOM SEED & ENVIRONMENT ============
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import random
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.set_visible_devices([], 'GPU')

# ============ CONFIG ============
DATASET = "D"
DATE = "13082026"
CV_FOLDS = 5
EPOCHS_FINAL = 100
PATIENCE_ES = 5
PATIENCE_LR = 5

# ============ MODEL SELECTION ============
# Choose which models to run: 'MLP', 'LinReg', 'LogReg', 'SVM', 'XGBoost', 'LightGBM', 'CatBoost'
# Order: MLP → Linear Regression → Logistic Regression → SVM → XGBoost → LightGBM → CatBoost
MODELS_TO_RUN = ['MLP', 'LinReg', 'LogReg', 'SVM', 'XGBoost', 'LightGBM', 'CatBoost']

# ============ INPUT FILE (Part 1 Results - FIXED) ============
FILE_RESULTS = "./0. D_CryptoGuard-MLP_GridSearchCV_5Fold_08082026/1.1_D_CryptoGuard-MLP_GridSearchCV_5Fold_Results_08082026.csv"

# ============ OUTPUT DIRECTORY & FILES ============
TAG = f"{DATASET}_ComparativeAnalysis"
OUT_DIR = f"./0. {TAG}_{DATE}"
os.makedirs(OUT_DIR, exist_ok=True)

FILE_LOSS_PLOT = f"{OUT_DIR}/0.1_{DATASET}_ComparativeAnalysis_Loss_{DATE}.png"
FILE_ACC_PLOT  = f"{OUT_DIR}/0.2_{DATASET}_ComparativeAnalysis_Accuracy_{DATE}.png"
FILE_METRICS_MASTER = f"{OUT_DIR}/0.3_{DATASET}_ComparativeAnalysis_MasterResults_{DATE}.csv"
FILE_METRICS_AGGS = f"{OUT_DIR}/0.4_{DATASET}_ComparativeAnalysis_AggregateResults_{DATE}.csv"

# ============ HELPER FUNCTIONS ============
def print_header(title):
    print("\n" + "="*80)
    print(title)
    print("="*80)

def print_step(step_num, title):
    print(f"\n[STEP {step_num}] {title}")
    print("-" * 80)

def calculate_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    
    return {
        'TP': int(tp), 
        'FP': int(fp), 
        'FN': int(fn), 
        'TN': int(tn),
        'Accuracy': f"{acc*100:.2f}%", 
        'Precision': f"{prec*100:.2f}%", 
        'Recall': f"{rec*100:.2f}%", 
        'F-measure': f"{f1*100:.2f}%",
        'MCC': f"{mcc:.4f}",
        'FPR': f"{fpr*100:.2f}%", 
        'FNR': f"{fnr*100:.2f}%"
    }

def create_mlp_model(hidden_layer1, hidden_layer2, hidden_layer3, 
                     dropout_rate, l2_reg, learning_rate, input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(hidden_layer1, activation='relu', kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        Dense(hidden_layer2, activation='relu', kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        Dense(hidden_layer3, activation='relu', kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid', kernel_regularizer=regularizers.l2(l2_reg))
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

def create_linreg_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(1, activation='linear')
    ])
    model.compile(
        loss='mean_squared_error',
        optimizer=keras.optimizers.SGD(learning_rate=0.1),
        metrics=['accuracy']
    )
    return model

def create_logreg_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(1, activation='sigmoid', 
              kernel_regularizer=regularizers.l2(0.5))
    ])
    model.compile(
        loss='binary_crossentropy',
        optimizer=keras.optimizers.SGD(learning_rate=0.01),
        metrics=['accuracy']
    )
    return model

def get_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=PATIENCE_ES, 
                     restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, 
                         patience=PATIENCE_LR, min_lr=1e-6, verbose=0)
    ]

# ============ MAIN EXECUTION ============
if __name__ == "__main__":
    print_header("COMPARATIVE ANALYSIS: CRYPTOGUARD-MLP vs LINEAR REGRESSION vs LOGISTIC REGRESSION")

    # ============ STEP 1: LOAD PART 1 GRIDSEARCH RESULTS ============
    print_step(1, "LOADING PART 1 GRIDSEARCH RESULTS")
    
    df_results = pd.read_csv(FILE_RESULTS)
    best_row = df_results.iloc[0]
    
    h1 = int(best_row['param_hidden_layer1'])
    h2 = int(best_row['param_hidden_layer2'])
    h3 = int(best_row['param_hidden_layer3'])
    dr = float(best_row['param_dropout_rate'])
    l2r = float(best_row['param_l2_reg'])
    lr = float(best_row['param_learning_rate'])
    bs = int(best_row['param_batch_size'])
    
    cv_train_mcc = float(best_row['mean_train_score'])
    cv_val_mcc = float(best_row['mean_test_score'])
    cv_gap_mcc = float(best_row['Gap_MCC'])
    rank = int(best_row['rank_test_score'])
    
    print(f"\n✓ Strategy 1: Best MCC (Rank {rank})")
    print(f"  Architecture      : {h1}-{h2}-{h3}")
    print(f"  Dropout Rate      : {dr}")
    print(f"  L2 Regularization : {l2r}")
    print(f"  Learning Rate     : {lr}")
    print(f"  Batch Size        : {bs}")
    print(f"  Mean Train Score  : {cv_train_mcc:.4f}")
    print(f"  Mean Test Score   : {cv_val_mcc:.4f}")
    print(f"  Gap MCC           : {cv_gap_mcc:.4f}")

    # ============ STEP 2: LOAD FULL DATASET ============
    print_step(2, "LOADING FULL DATASET")
    
    df = pd.read_excel(f'./Datasets/{DATASET}.xlsx')
    X = df.drop(columns=['CLASS_LABEL'])
    y = df['CLASS_LABEL']
    input_dim = X.shape[1]
    
    print(f"\n✓ X shape: {X.shape}")
    print(f"✓ y shape: {y.shape}")
    print(f"✓ Input dimension: {input_dim}")
    print(f"✓ Class distribution: {y.value_counts().to_dict()}")

    # ============ STEP 3: 5-FOLD STRATIFIED CV (FOLD BY FOLD) ============
    print_step(3, "5-FOLD STRATIFIED CV (MLP → LINEAR REGRESSION → LOGISTIC REGRESSION → SVM → XGBoost → LightGBM → CatBoost)")
    
    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
    fold_metrics_list = []
    fold_histories_mlp = []
    fold_histories_linreg = []
    fold_histories_logreg = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n[Fold {fold}/{CV_FOLDS}]")
        
        # ========== Split 1: Outer fold (train 80%, test 20%) ==========
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx].values.astype(np.float32)
        y_test = y.iloc[test_idx].values.astype(np.float32)
        
        # ========== Split 2: Inner stratified split (10% validation from training) ==========
        X_train_real, X_val, y_train_real, y_val = train_test_split(
            X_train, y_train,
            test_size=0.10,
            random_state=SEED,
            stratify=y_train
        )
        
        # ========== Scale (fit on actual training, transform val and test) ==========
        scaler = StandardScaler()
        X_train_real_sc = scaler.fit_transform(X_train_real).astype(np.float32)
        X_val_sc = scaler.transform(X_val).astype(np.float32)
        X_test_sc = scaler.transform(X_test).astype(np.float32)
        
        # ========== MODEL 1: CRYPTOGUARD-MLP (PART 2A SETUP) ==========
        if 'MLP' in MODELS_TO_RUN:
            print(f"  Training CryptoGuard-MLP...", end='', flush=True)
            model_mlp = create_mlp_model(h1, h2, h3, dr, l2r, lr, input_dim)
            history_mlp = model_mlp.fit(
                X_train_real_sc, y_train_real,
                epochs=EPOCHS_FINAL,
                batch_size=bs,
                validation_data=(X_val_sc, y_val),
                callbacks=get_callbacks(),
                verbose=0
            )
            epochs_trained_mlp = len(history_mlp.history['loss'])
            fold_histories_mlp.append(history_mlp.history)
            y_pred_mlp = (model_mlp.predict(X_test_sc, verbose=0, batch_size=bs) > 0.5).astype(int).flatten()
            metrics_mlp = calculate_metrics(y_test, y_pred_mlp)
            metrics_mlp['Epochs_Trained'] = epochs_trained_mlp
            fold_metrics_list.append({'Fold': f'Fold {fold}', 'Model': 'CryptoGuard-MLP', **metrics_mlp})
            print(f" ✓ Acc={metrics_mlp['Accuracy']}, MCC={metrics_mlp['MCC']}, Epochs={epochs_trained_mlp}")
            del model_mlp, history_mlp
            tf.keras.backend.clear_session()
            gc.collect()
        
        # ========== MODEL 2: LINEAR REGRESSION ==========
        if 'LinReg' in MODELS_TO_RUN:
            print(f"  Training Linear Regression...", end='', flush=True)
            model_linreg = create_linreg_model(input_dim)
            history_linreg = model_linreg.fit(
                X_train_real_sc, y_train_real,
                epochs=EPOCHS_FINAL,
                batch_size=bs,
                validation_data=(X_val_sc, y_val),
                callbacks=get_callbacks(),
                verbose=0
            )
            epochs_trained_linreg = len(history_linreg.history['loss'])
            fold_histories_linreg.append(history_linreg.history)
            y_pred_linreg = np.where(model_linreg.predict(X_test_sc, verbose=0) > 0.5, 1, 0).flatten()
            metrics_linreg = calculate_metrics(y_test, y_pred_linreg)
            metrics_linreg['Epochs_Trained'] = epochs_trained_linreg
            fold_metrics_list.append({'Fold': f'Fold {fold}', 'Model': 'Linear Regression', **metrics_linreg})
            print(f" ✓ Acc={metrics_linreg['Accuracy']}, MCC={metrics_linreg['MCC']}, Epochs={epochs_trained_linreg}")
            del model_linreg, history_linreg
            tf.keras.backend.clear_session()
            gc.collect()
        
        # ========== MODEL 3: LOGISTIC REGRESSION ==========
        if 'LogReg' in MODELS_TO_RUN:
            print(f"  Training Logistic Regression...", end='', flush=True)
            model_logreg = create_logreg_model(input_dim)
            history_logreg = model_logreg.fit(
                X_train_real_sc, y_train_real,
                epochs=EPOCHS_FINAL,
                batch_size=bs,
                validation_data=(X_val_sc, y_val),
                callbacks=get_callbacks(),
                verbose=0
            )
            epochs_trained_logreg = len(history_logreg.history['loss'])
            fold_histories_logreg.append(history_logreg.history)
            y_pred_logreg = (model_logreg.predict(X_test_sc, verbose=0, batch_size=bs) > 0.5).astype(int).flatten()
            metrics_logreg = calculate_metrics(y_test, y_pred_logreg)
            metrics_logreg['Epochs_Trained'] = epochs_trained_logreg
            fold_metrics_list.append({'Fold': f'Fold {fold}', 'Model': 'Logistic Regression', **metrics_logreg})
            print(f" ✓ Acc={metrics_logreg['Accuracy']}, MCC={metrics_logreg['MCC']}, Epochs={epochs_trained_logreg}")
            del model_logreg, history_logreg
            tf.keras.backend.clear_session()
            gc.collect()
        
        # ========== MODEL 4: SVM ==========
        if 'SVM' in MODELS_TO_RUN:
            print(f"  Training SVM...", end='', flush=True)
            svm_model = SVC(kernel='rbf', C=0.15, random_state=SEED)
            svm_model.fit(X_train_real_sc, y_train_real)
            y_pred_svm = svm_model.predict(X_test_sc)
            metrics_svm = calculate_metrics(y_test, y_pred_svm)
            metrics_svm['Epochs_Trained'] = '-'
            fold_metrics_list.append({'Fold': f'Fold {fold}', 'Model': 'SVM', **metrics_svm})
            print(f" ✓ Acc={metrics_svm['Accuracy']}, MCC={metrics_svm['MCC']}")
            del svm_model
            gc.collect()
        
        # ========== MODEL 5: XGBOOST (WITH EARLY STOPPING) ==========
        if 'XGBoost' in MODELS_TO_RUN:
            print(f"  Training XGBoost...", end='', flush=True)
            xgb_model = XGBClassifier(
                n_estimators=100, learning_rate=0.01,
                max_depth=6, eval_metric='logloss',
                early_stopping_rounds=PATIENCE_ES,
                random_state=SEED, verbosity=0)
            xgb_model.fit(
                X_train_real_sc, y_train_real,
                eval_set=[(X_val_sc, y_val)],
                verbose=False)
            y_pred_xgb = xgb_model.predict(X_test_sc)
            metrics_xgb = calculate_metrics(y_test, y_pred_xgb)
            metrics_xgb['Epochs_Trained'] = '-'
            fold_metrics_list.append({'Fold': f'Fold {fold}', 'Model': 'XGBoost', **metrics_xgb})
            print(f" ✓ Acc={metrics_xgb['Accuracy']}, MCC={metrics_xgb['MCC']}")
            del xgb_model
            gc.collect()
        
        # ========== MODEL 6: LIGHTGBM (WITH EARLY STOPPING) ==========
        if 'LightGBM' in MODELS_TO_RUN:
            print(f"  Training LightGBM...", end='', flush=True)
            lgb_model = LGBMClassifier(
                n_estimators=100, learning_rate=0.01,
                max_depth=6, random_state=SEED, verbosity=-1)
            lgb_model.fit(
                X_train_real_sc, y_train_real,
                eval_set=[(X_val_sc, y_val)],
                callbacks=[lgb.early_stopping(PATIENCE_ES)])
            y_pred_lgb = lgb_model.predict(X_test_sc)
            metrics_lgb = calculate_metrics(y_test, y_pred_lgb)
            metrics_lgb['Epochs_Trained'] = '-'
            fold_metrics_list.append({'Fold': f'Fold {fold}', 'Model': 'LightGBM', **metrics_lgb})
            print(f" ✓ Acc={metrics_lgb['Accuracy']}, MCC={metrics_lgb['MCC']}")
            del lgb_model
            gc.collect()
        
        # ========== MODEL 7: CATBOOST (WITH EARLY STOPPING) ==========
        if 'CatBoost' in MODELS_TO_RUN:
            print(f"  Training CatBoost...", end='', flush=True)
            cat_model = CatBoostClassifier(
                iterations=100, learning_rate=0.01,
                depth=6, early_stopping_rounds=PATIENCE_ES,
                random_state=SEED, verbose=0)
            cat_model.fit(
                X_train_real_sc, y_train_real,
                eval_set=(X_val_sc, y_val))
            y_pred_cat = cat_model.predict(X_test_sc)
            metrics_cat = calculate_metrics(y_test, y_pred_cat)
            metrics_cat['Epochs_Trained'] = '-'
            fold_metrics_list.append({'Fold': f'Fold {fold}', 'Model': 'CatBoost', **metrics_cat})
            print(f" ✓ Acc={metrics_cat['Accuracy']}, MCC={metrics_cat['MCC']}")
            del cat_model
            gc.collect()
        
        # Clean up
        del X_train, X_test, X_train_real, X_val, X_train_real_sc, X_val_sc, X_test_sc
        gc.collect()

    # ============ STEP 4: SAVE RESULTS ============
    print_step(4, "SAVING CSV RESULTS")
    
    fold_metrics_df = pd.DataFrame(fold_metrics_list)
    
    # ========== MASTERRESULTS (All folds + Mean per model) ==========
    # Order: MLP → Linear Regression → Logistic Regression → SVM → XGBoost → LightGBM → CatBoost
    # Format: Model, Fold, Epochs_Trained, TP, FP, FN, TN, Accuracy, FPR, FNR, MCC, Precision, Recall, F-measure
    master_rows = []
    
    for model_name in ['CryptoGuard-MLP', 'Linear Regression', 'Logistic Regression', 'SVM', 'XGBoost', 'LightGBM', 'CatBoost']:
        if model_name == 'CryptoGuard-MLP' and 'MLP' not in MODELS_TO_RUN:
            continue
        if model_name == 'Linear Regression' and 'LinReg' not in MODELS_TO_RUN:
            continue
        if model_name == 'Logistic Regression' and 'LogReg' not in MODELS_TO_RUN:
            continue
        if model_name == 'SVM' and 'SVM' not in MODELS_TO_RUN:
            continue
        if model_name == 'XGBoost' and 'XGBoost' not in MODELS_TO_RUN:
            continue
        if model_name == 'LightGBM' and 'LightGBM' not in MODELS_TO_RUN:
            continue
        if model_name == 'CatBoost' and 'CatBoost' not in MODELS_TO_RUN:
            continue
        
        # Get all folds for this model (5 rows)
        model_folds = fold_metrics_df[fold_metrics_df['Model'] == model_name]
        
        # Add 5 fold rows
        for idx, row in model_folds.iterrows():
            master_rows.append(row.to_dict())
        
        # Calculate and add Mean row (NO TP/FP/FN/TN/Epochs_Trained)
        accuracy_vals = [float(x.strip('%')) for x in model_folds['Accuracy']]
        fpr_vals = [float(x.strip('%')) for x in model_folds['FPR']]
        fnr_vals = [float(x.strip('%')) for x in model_folds['FNR']]
        mcc_vals = [float(x) for x in model_folds['MCC']]
        precision_vals = [float(x.strip('%')) for x in model_folds['Precision']]
        recall_vals = [float(x.strip('%')) for x in model_folds['Recall']]
        f_measure_vals = [float(x.strip('%')) for x in model_folds['F-measure']]
        
        mean_row = {
            'Model': model_name,
            'Fold': 'Mean',
            'Epochs_Trained': '-',
            'TP': '-',
            'FP': '-',
            'FN': '-',
            'TN': '-',
            'Accuracy': f"{np.mean(accuracy_vals):.2f}±{np.std(accuracy_vals):.2f}%",
            'FPR': f"{np.mean(fpr_vals):.2f}±{np.std(fpr_vals):.2f}%",
            'FNR': f"{np.mean(fnr_vals):.2f}±{np.std(fnr_vals):.2f}%",
            'MCC': f"{np.mean(mcc_vals):.4f}±{np.std(mcc_vals):.4f}",
            'Precision': f"{np.mean(precision_vals):.2f}±{np.std(precision_vals):.2f}%",
            'Recall': f"{np.mean(recall_vals):.2f}±{np.std(recall_vals):.2f}%",
            'F-measure': f"{np.mean(f_measure_vals):.2f}±{np.std(f_measure_vals):.2f}%"
        }
        master_rows.append(mean_row)
    
    master_df = pd.DataFrame(master_rows)
    
    # Reorder columns to exact order requested
    ordered_cols = ['Model', 'Fold', 'Epochs_Trained', 'TP', 'FP', 'FN', 'TN', 'Accuracy', 'FPR', 'FNR', 'MCC', 'Precision', 'Recall', 'F-measure']
    master_df = master_df[ordered_cols]
    
    master_df.to_csv(FILE_METRICS_MASTER, index=False)
    print(f"\n✓ Saved MasterResults: {FILE_METRICS_MASTER}")
    
    # ========== AGGREGATERESULTS (Summary by model - NO EPOCHS_TRAINED) ==========
    # Order: MLP → Linear Regression → Logistic Regression → SVM → XGBoost → LightGBM → CatBoost
    agg_rows = []
    for model_name in ['CryptoGuard-MLP', 'Linear Regression', 'Logistic Regression', 'SVM', 'XGBoost', 'LightGBM', 'CatBoost']:
        if model_name == 'CryptoGuard-MLP' and 'MLP' not in MODELS_TO_RUN:
            continue
        if model_name == 'Linear Regression' and 'LinReg' not in MODELS_TO_RUN:
            continue
        if model_name == 'Logistic Regression' and 'LogReg' not in MODELS_TO_RUN:
            continue
        if model_name == 'SVM' and 'SVM' not in MODELS_TO_RUN:
            continue
        if model_name == 'XGBoost' and 'XGBoost' not in MODELS_TO_RUN:
            continue
        if model_name == 'LightGBM' and 'LightGBM' not in MODELS_TO_RUN:
            continue
        if model_name == 'CatBoost' and 'CatBoost' not in MODELS_TO_RUN:
            continue
            
        model_data = fold_metrics_df[fold_metrics_df['Model'] == model_name]
        
        accuracy_vals = [float(x.strip('%')) for x in model_data['Accuracy']]
        fpr_vals = [float(x.strip('%')) for x in model_data['FPR']]
        fnr_vals = [float(x.strip('%')) for x in model_data['FNR']]
        mcc_vals = [float(x) for x in model_data['MCC']]
        precision_vals = [float(x.strip('%')) for x in model_data['Precision']]
        recall_vals = [float(x.strip('%')) for x in model_data['Recall']]
        f_measure_vals = [float(x.strip('%')) for x in model_data['F-measure']]
       
        agg_row = {
            'Model': model_name,
            'Accuracy': f"{np.mean(accuracy_vals):.2f}±{np.std(accuracy_vals):.2f}%",
            'FPR': f"{np.mean(fpr_vals):.2f}±{np.std(fpr_vals):.2f}%",
            'FNR': f"{np.mean(fnr_vals):.2f}±{np.std(fnr_vals):.2f}%",
            'MCC': f"{np.mean(mcc_vals):.4f}±{np.std(mcc_vals):.4f}",
            'Precision': f"{np.mean(precision_vals):.2f}±{np.std(precision_vals):.2f}%",
            'Recall': f"{np.mean(recall_vals):.2f}±{np.std(recall_vals):.2f}%",
            'F-measure': f"{np.mean(f_measure_vals):.2f}±{np.std(f_measure_vals):.2f}%",
        }
        agg_rows.append(agg_row)
    
    agg_df = pd.DataFrame(agg_rows)
    
    # Reorder columns to exact order
    agg_cols_ordered = ['Model', 'Accuracy', 'FPR', 'FNR', 'MCC', 'Precision', 'Recall', 'F-measure']
    agg_df = agg_df[agg_cols_ordered]
    
    agg_df.to_csv(FILE_METRICS_AGGS, index=False)
    print(f"✓ Saved AggregateResults: {FILE_METRICS_AGGS}")

    # ============ STEP 5: GENERATE COMBINED CURVES ============
    print_step(5, f"GENERATING COMBINED TRAINING CURVES ({', '.join(MODELS_TO_RUN)})")
    
    # ========== Prepare MLP data ==========
    if 'MLP' in MODELS_TO_RUN:
        min_epochs_mlp = min([len(h['loss']) for h in fold_histories_mlp])
        print(f"\n✓ MLP min_epochs: {min_epochs_mlp}")
        
        train_loss_mlp = np.array([h['loss'][:min_epochs_mlp] for h in fold_histories_mlp])
        val_loss_mlp = np.array([h['val_loss'][:min_epochs_mlp] for h in fold_histories_mlp])
        train_acc_mlp = np.array([h['accuracy'][:min_epochs_mlp] for h in fold_histories_mlp])
        val_acc_mlp = np.array([h['val_accuracy'][:min_epochs_mlp] for h in fold_histories_mlp])
        
        train_loss_mean_mlp = np.mean(train_loss_mlp, axis=0)
        train_loss_std_mlp = np.std(train_loss_mlp, axis=0)
        val_loss_mean_mlp = np.mean(val_loss_mlp, axis=0)
        val_loss_std_mlp = np.std(val_loss_mlp, axis=0)
        
        train_acc_mean_mlp = np.mean(train_acc_mlp, axis=0)
        train_acc_std_mlp = np.std(train_acc_mlp, axis=0)
        val_acc_mean_mlp = np.mean(val_acc_mlp, axis=0)
        val_acc_std_mlp = np.std(val_acc_mlp, axis=0)
        
        epochs_mlp = np.arange(1, len(train_loss_mean_mlp) + 1)
    
    # ========== Prepare Linear Regression data ==========
    if 'LinReg' in MODELS_TO_RUN:
        min_epochs_linreg = min([len(h['loss']) for h in fold_histories_linreg])
        print(f"✓ LinReg min_epochs: {min_epochs_linreg}")
        
        train_loss_linreg = np.array([h['loss'][:min_epochs_linreg] for h in fold_histories_linreg])
        val_loss_linreg = np.array([h['val_loss'][:min_epochs_linreg] for h in fold_histories_linreg])
        train_acc_linreg = np.array([h['accuracy'][:min_epochs_linreg] for h in fold_histories_linreg])
        val_acc_linreg = np.array([h['val_accuracy'][:min_epochs_linreg] for h in fold_histories_linreg])
        
        train_loss_mean_linreg = np.mean(train_loss_linreg, axis=0)
        train_loss_std_linreg = np.std(train_loss_linreg, axis=0)
        val_loss_mean_linreg = np.mean(val_loss_linreg, axis=0)
        val_loss_std_linreg = np.std(val_loss_linreg, axis=0)
        
        train_acc_mean_linreg = np.mean(train_acc_linreg, axis=0)
        train_acc_std_linreg = np.std(train_acc_linreg, axis=0)
        val_acc_mean_linreg = np.mean(val_acc_linreg, axis=0)
        val_acc_std_linreg = np.std(val_acc_linreg, axis=0)
        
        epochs_linreg = np.arange(1, len(train_loss_mean_linreg) + 1)
    
    # ========== Prepare Logistic Regression data ==========
    if 'LogReg' in MODELS_TO_RUN:
        min_epochs_logreg = min([len(h['loss']) for h in fold_histories_logreg])
        print(f"✓ LogReg min_epochs: {min_epochs_logreg}")
        
        train_loss_logreg = np.array([h['loss'][:min_epochs_logreg] for h in fold_histories_logreg])
        val_loss_logreg = np.array([h['val_loss'][:min_epochs_logreg] for h in fold_histories_logreg])
        train_acc_logreg = np.array([h['accuracy'][:min_epochs_logreg] for h in fold_histories_logreg])
        val_acc_logreg = np.array([h['val_accuracy'][:min_epochs_logreg] for h in fold_histories_logreg])
        
        train_loss_mean_logreg = np.mean(train_loss_logreg, axis=0)
        train_loss_std_logreg = np.std(train_loss_logreg, axis=0)
        val_loss_mean_logreg = np.mean(val_loss_logreg, axis=0)
        val_loss_std_logreg = np.std(val_loss_logreg, axis=0)
        
        train_acc_mean_logreg = np.mean(train_acc_logreg, axis=0)
        train_acc_std_logreg = np.std(train_acc_logreg, axis=0)
        val_acc_mean_logreg = np.mean(val_acc_logreg, axis=0)
        val_acc_std_logreg = np.std(val_acc_logreg, axis=0)
        
        epochs_logreg = np.arange(1, len(train_loss_mean_logreg) + 1)
    
    # ========== PLOT 1: COMBINED LOSS (MLP, LinReg, LogReg only) ==========
    fig, ax = plt.subplots(figsize=(12, 6))
    
    if 'MLP' in MODELS_TO_RUN:
        ax.plot(epochs_mlp, train_loss_mean_mlp, label='CryptoGuard-MLP_Train', color='#228B22', linewidth=2)
        ax.fill_between(epochs_mlp, train_loss_mean_mlp - train_loss_std_mlp, train_loss_mean_mlp + train_loss_std_mlp, color='#228B22', alpha=0.2)
        ax.plot(epochs_mlp, val_loss_mean_mlp, label='CryptoGuard-MLP_Val', color='#228B22', linewidth=2, linestyle='--')
        ax.fill_between(epochs_mlp, val_loss_mean_mlp - val_loss_std_mlp, val_loss_mean_mlp + val_loss_std_mlp, color='#228B22', alpha=0.2)
    
    if 'LinReg' in MODELS_TO_RUN:
        ax.plot(epochs_linreg, train_loss_mean_linreg, label='LinReg_Train', color='#FF0000', linewidth=2)
        ax.fill_between(epochs_linreg, train_loss_mean_linreg - train_loss_std_linreg, train_loss_mean_linreg + train_loss_std_linreg, color='#FF0000', alpha=0.2)
        ax.plot(epochs_linreg, val_loss_mean_linreg, label='LinReg_Val', color='#FF0000', linewidth=2, linestyle='--')
        ax.fill_between(epochs_linreg, val_loss_mean_linreg - val_loss_std_linreg, val_loss_mean_linreg + val_loss_std_linreg, color='#FF0000', alpha=0.2)
    
    if 'LogReg' in MODELS_TO_RUN:
        ax.plot(epochs_logreg, train_loss_mean_logreg, label='LogReg_Train', color='#FF7F00', linewidth=2)
        ax.fill_between(epochs_logreg, train_loss_mean_logreg - train_loss_std_logreg, train_loss_mean_logreg + train_loss_std_logreg, color='#FF7F00', alpha=0.2)
        ax.plot(epochs_logreg, val_loss_mean_logreg, label='LogReg_Val', color='#FF7F00', linewidth=2, linestyle='--')
        ax.fill_between(epochs_logreg, val_loss_mean_logreg - val_loss_std_logreg, val_loss_mean_logreg + val_loss_std_logreg, color='#FF7F00', alpha=0.2)
    
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title('Combined Training Loss (5-Fold CV Mean ± Std)', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=12, shadow=True, frameon=True)
    ax.grid(False)
    plt.tight_layout()
    plt.savefig(FILE_LOSS_PLOT, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {FILE_LOSS_PLOT}")
    plt.close()
    
    # ========== PLOT 2: COMBINED ACCURACY (MLP, LinReg, LogReg only) ==========
    fig, ax = plt.subplots(figsize=(12, 6))
    
    if 'MLP' in MODELS_TO_RUN:
        ax.plot(epochs_mlp, train_acc_mean_mlp, label='MLP Training Accuracy', color='#228B22', linewidth=2)
        ax.fill_between(epochs_mlp, train_acc_mean_mlp - train_acc_std_mlp, train_acc_mean_mlp + train_acc_std_mlp, color='#228B22', alpha=0.2)
        ax.plot(epochs_mlp, val_acc_mean_mlp, label='MLP Validation Accuracy', color='#228B22', linewidth=2, linestyle='--')
        ax.fill_between(epochs_mlp, val_acc_mean_mlp - val_acc_std_mlp, val_acc_mean_mlp + val_acc_std_mlp, color='#228B22', alpha=0.2)
    
    if 'LinReg' in MODELS_TO_RUN:
        ax.plot(epochs_linreg, train_acc_mean_linreg, label='Linear Regression Training Accuracy', color='#FF0000', linewidth=2)
        ax.fill_between(epochs_linreg, train_acc_mean_linreg - train_acc_std_linreg, train_acc_mean_linreg + train_acc_std_linreg, color='#FF0000', alpha=0.2)
        ax.plot(epochs_linreg, val_acc_mean_linreg, label='Linear Regression Validation Accuracy', color='#FF0000', linewidth=2, linestyle='--')
        ax.fill_between(epochs_linreg, val_acc_mean_linreg - val_acc_std_linreg, val_acc_mean_linreg + val_acc_std_linreg, color='#FF0000', alpha=0.2)
    
    if 'LogReg' in MODELS_TO_RUN:
        ax.plot(epochs_logreg, train_acc_mean_logreg, label='Logistic Regression Training Accuracy', color='#FF7F00', linewidth=2)
        ax.fill_between(epochs_logreg, train_acc_mean_logreg - train_acc_std_logreg, train_acc_mean_logreg + train_acc_std_logreg, color='#FF7F00', alpha=0.2)
        ax.plot(epochs_logreg, val_acc_mean_logreg, label='Logistic Regression Validation Accuracy', color='#FF7F00', linewidth=2, linestyle='--')
        ax.fill_between(epochs_logreg, val_acc_mean_logreg - val_acc_std_logreg, val_acc_mean_logreg + val_acc_std_logreg, color='#FF7F00', alpha=0.2)
    
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title('Combined Training Accuracy (5-Fold CV Mean ± Std)', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=12, shadow=True, frameon=True)
    ax.grid(False)
    plt.tight_layout()
    plt.savefig(FILE_ACC_PLOT, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {FILE_ACC_PLOT}")
    plt.close()

    # ========== SUMMARY ==========
    print("\n" + "="*80)
    print("COMPARATIVE ANALYSIS COMPLETE!")
    print("="*80)
    
    print(f"\nModels Executed: {', '.join(MODELS_TO_RUN)}")
    print(f"\nOutput Files:")
    print(f"  ✓ {FILE_LOSS_PLOT}")
    if 'MLP' in MODELS_TO_RUN:
        print(f"     GREEN (#228B22): CryptoGuard-MLP (solid=train, dashed=val)")
    if 'LinReg' in MODELS_TO_RUN:
        print(f"     RED (#FF0000): Linear Regression (solid=train, dashed=val)")
    if 'LogReg' in MODELS_TO_RUN:
        print(f"     ORANGE (#FF7F00): Logistic Regression (solid=train, dashed=val)")
    print(f"  ✓ {FILE_ACC_PLOT}")
    if 'MLP' in MODELS_TO_RUN:
        print(f"     GREEN (#228B22): CryptoGuard-MLP (solid=train, dashed=val)")
    if 'LinReg' in MODELS_TO_RUN:
        print(f"     RED (#FF0000): Linear Regression (solid=train, dashed=val)")
    if 'LogReg' in MODELS_TO_RUN:
        print(f"     ORANGE (#FF7F00): Logistic Regression (solid=train, dashed=val)")
    print(f"  ✓ {FILE_METRICS_MASTER}")
    print(f"  ✓ {FILE_METRICS_AGGS}")
    print(f"\nOutput Directory: {OUT_DIR}")


COMPARATIVE ANALYSIS: CRYPTOGUARD-MLP vs LINEAR REGRESSION vs LOGISTIC REGRESSION

[STEP 1] LOADING PART 1 GRIDSEARCH RESULTS
--------------------------------------------------------------------------------

✓ Strategy 1: Best MCC (Rank 1)
  Architecture      : 768-512-128
  Dropout Rate      : 0.5
  L2 Regularization : 0.005
  Learning Rate     : 0.0001
  Batch Size        : 64
  Mean Train Score  : 0.9075
  Mean Test Score   : 0.8689
  Gap MCC           : 0.0386

[STEP 2] LOADING FULL DATASET
--------------------------------------------------------------------------------

✓ X shape: (20000, 525)
✓ y shape: (20000,)
✓ Input dimension: 525
✓ Class distribution: {0: 10000, 1: 10000}

[STEP 3] 5-FOLD STRATIFIED CV (MLP → LINEAR REGRESSION → LOGISTIC REGRESSION → SVM → XGBoost → LightGBM → CatBoost)
--------------------------------------------------------------------------------

[Fold 1/5]
  Training CryptoGuard-MLP... ✓ Acc=93.03%, MCC=0.8606, Epochs=65
  Training Linear Regression...

In [5]:
import pandas as pd
import re

# Read aggregate results
df = pd.read_csv("./1. D_ComparativeAnalysis_13082026/1.4_D_ComparativeAnalysis_AggregateResults_13082026.csv")

# Function to extract mean and std from "XX.XX±X.XX%"
def extract_mean_std(value):
    match = re.search(r'([\d.]+)±([\d.]+)', str(value))
    if match:
        mean = float(match.group(1))
        std = float(match.group(2))
        return mean, std
    return None, None

# Create new dataframe
data = {'Model': df['Model']}

# Sheet 1: Accuracy, FPR, FNR, MCC
for metric in ['Accuracy', 'FPR', 'FNR', 'MCC']:
    means = []
    stds = []
    for val in df[metric]:
        mean, std = extract_mean_std(val)
        if mean is not None:
            # Convert percentage to 0-1 (except MCC which is already 0-1)
            if metric != 'MCC' and mean > 1:
                mean = mean / 100
                std = std / 100
            means.append(mean)
            stds.append(std)
    
    data[f'MEAN_{metric}'] = means
    data[f'SD_{metric}'] = stds

sheet1 = pd.DataFrame(data)[['Model', 'MEAN_Accuracy', 'SD_Accuracy', 'MEAN_FPR', 'SD_FPR', 'MEAN_FNR', 'SD_FNR', 'MEAN_MCC', 'SD_MCC']]

# Sheet 2: Precision, Recall, F-measure
data2 = {'Model': df['Model']}

for metric in ['Precision', 'Recall', 'F-measure']:
    means = []
    stds = []
    for val in df[metric]:
        mean, std = extract_mean_std(val)
        if mean is not None:
            # Convert percentage to 0-1
            if mean > 1:
                mean = mean / 100
                std = std / 100
            means.append(mean)
            stds.append(std)
    
    data2[f'MEAN_{metric}'] = means
    data2[f'SD_{metric}'] = stds

sheet2 = pd.DataFrame(data2)[['Model', 'MEAN_Precision', 'SD_Precision', 'MEAN_Recall', 'SD_Recall', 'MEAN_F-measure', 'SD_F-measure']]

# Save to Excel with 2 sheets
with pd.ExcelWriter("./1. D_ComparativeAnalysis_13082026/1.5_D_BarChartData_13082026.xlsx", engine='openpyxl') as writer:
    sheet1.to_excel(writer, sheet_name='Chart_1', index=False)
    sheet2.to_excel(writer, sheet_name='Chart_2', index=False)

print("✅ Done!")
print("\n--- Sheet 1 (Chart_1) ---")
print(sheet1)
print("\n--- Sheet 2 (Chart_2) ---")
print(sheet2)

✅ Done!

--- Sheet 1 (Chart_1) ---
                 Model  MEAN_Accuracy  SD_Accuracy  MEAN_FPR  SD_FPR  \
0      CryptoGuard-MLP         0.9301       0.0037    0.0662  0.0112   
1    Linear Regression         0.4762       0.0873    0.5085  0.1486   
2  Logistic Regression         0.8639       0.0030    0.2008  0.0082   
3                  SVM         0.8879       0.0059    0.1120  0.0177   
4              XGBoost         0.9185       0.0039    0.0671  0.0089   
5             LightGBM         0.9190       0.0031    0.0653  0.0067   
6             CatBoost         0.9207       0.0044    0.0511  0.0058   

   MEAN_FNR  SD_FNR  MEAN_MCC  SD_MCC  
0    0.0736  0.0114    0.8604  0.0075  
1    0.5390  0.3097    0.0441  0.2026  
2    0.0714  0.0066    0.7341  0.0058  
3    0.1122  0.0093    0.7760  0.0117  
4    0.0959  0.0078    0.8374  0.0079  
5    0.0967  0.0067    0.8385  0.0061  
6    0.1075  0.0107    0.8428  0.0082  

--- Sheet 2 (Chart_2) ---
                 Model  MEAN_Precision  S

In [ ]:
# Standalone Sklearn LogisticRegression Test

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix
import pandas as pd
import numpy as np

# ============ LOAD DATA ============
file_path = './Datasets/D.xlsx'
df = pd.read_excel(file_path)
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

print(f"✓ Data loaded: X shape={X.shape}, y shape={y.shape}")

# ============ 5-FOLD CV ============
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_metrics = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n[Fold {fold}/5]")
    
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # Validation split
    X_train_real, X_val, y_train_real, y_val = train_test_split(
        X_train, y_train, test_size=0.10, random_state=42, stratify=y_train
    )
    
    # Scale
    scaler = StandardScaler()
    X_train_real_sc = scaler.fit_transform(X_train_real)
    X_val_sc = scaler.transform(X_val)
    X_test_sc = scaler.transform(X_test)
    
    # Train sklearn LogReg
    logreg = LogisticRegression(
        C=0.0001,
        penalty='l2',
        solver='lbfgs',
        max_iter=500,
        random_state=42
    )
    logreg.fit(X_train_real_sc, y_train_real)
    
    # Predict
    y_pred = logreg.predict(X_test_sc)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)
    
    print(f"  Acc={acc*100:.2f}%, MCC={mcc:.4f}")
    print(f"  Precision={prec*100:.2f}%, Recall={rec*100:.2f}%, F1={f1*100:.2f}%")
    
    fold_metrics.append({
        'Fold': fold,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'MCC': mcc,
        'FPR': fpr,
        'FNR': fnr
    })

# ============ SUMMARY ============
df_results = pd.DataFrame(fold_metrics)
print("\n" + "="*80)
print("SKLEARN LOGISTIC REGRESSION - 5-FOLD CV RESULTS")
print("="*80)
print(f"\nAccuracy:  {df_results['Accuracy'].mean()*100:.2f}±{df_results['Accuracy'].std()*100:.2f}%")
print(f"Precision: {df_results['Precision'].mean()*100:.2f}±{df_results['Precision'].std()*100:.2f}%")
print(f"Recall:    {df_results['Recall'].mean()*100:.2f}±{df_results['Recall'].std()*100:.2f}%")
print(f"F1-Score:  {df_results['F1'].mean()*100:.2f}±{df_results['F1'].std()*100:.2f}%")
print(f"MCC:       {df_results['MCC'].mean():.4f}±{df_results['MCC'].std():.4f}")

In [1]:
import numpy as np

accuracy_values = [93.03, 92.58, 92.83, 93.70, 92.92]

mean = np.mean(accuracy_values)
std = np.std(accuracy_values)

print(f"{mean:.2f}% ± {std:.2f}%")
# Output: 93.01% ± 0.37%

93.01% ± 0.37%
